In [2]:

print("\nBenchmark completato.")
print(f"Risultati salvati in: {OUT_JSONL}")
print_partial_summary(results, columns)



Benchmark completato.
Risultati salvati in: ../../data/test/benchmark_lmstudio_20251203_182902_synt_test_peft_quant.jsonl

=== Partial summary ===
                                                   codebleu  codebert_f1
model                              variant quant                        
tap_fc_deepseek_merge_6_7b_q4_k_m  base    Q4_K_M  0.422987     0.769455
tap_ft_codellama_merge_7b_q4_k_m   base    Q4_K_M  0.427253     0.826989
tap_ft_codestral_merge_7_3b_q4_k_m base    Q4_K_M  0.447198     0.838517
tap_ft_gemma_merge_8_5b_q4_k_m     base    Q4_K_M  0.247746     0.666845
tap_ft_qwen_merge_7_6b_q4_k_m      base    Q4_K_M  0.436850     0.846469



In [1]:
import os
import sys
import json
import time
import subprocess
from dataclasses import dataclass
from datetime import datetime
import requests
import pandas as pd
from tqdm.auto import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from codebleu import calc_codebleu
from code_bert_score import score as codebert_score
import re

# ------------------------------------------------------------
# CONFIG GENERALE
# ------------------------------------------------------------
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_JSONL = f"../../data/test/benchmark_lmstudio_{RUN_TS}_real_test_zs_quant_with_only_intent.jsonl"

# Server LM Studio REST (porta di default 1234)
LMSTUDIO_BASE_URL = "http://localhost:1234"

# quanti esempi usare (None = tutti)
TEST_SIZE = 4
LANG      = "javascript"

# stampa riepilogo dopo ogni modello
PRINT_EVERY_MODELS = 1

# ------------------------------------------------------------
# ROOT + LOADER DATASET (adattato dal tuo script)
# ------------------------------------------------------------
def detect_project_root():
    """
    Risale le directory finché trova una cartella 'src'.
    Modifica il path di partenza se serve.
    """
    root = r"C:\Users\ldesa\PycharmProjects\TAP basic\notebooks\fine_tuning"
    root = os.path.abspath(root)

    while True:
        if os.path.exists(os.path.join(root, "src")):
            break
        new_root = os.path.dirname(root)
        if new_root == root:
            raise RuntimeError("Non trovo la cartella 'src'. Modifica detect_project_root().")
        root = new_root

    if root not in sys.path:
        sys.path.append(root)
    print(f"Project root detected: {root}")
    return root

def load_df_real(limit=None):
    root = detect_project_root()
    real_path = os.path.join(root, "data", "dataset", "applets", "applets_real_new.jsonl")
    df = pd.read_json(real_path, lines=True, orient="records")
    if limit is not None:
        df = df[:limit]
    print(f"Loaded df_real shape={df.shape}")
    return df

# ------------------------------------------------------------
# ESTRAZIONE CODICE JS DA RISPOSTE (facoltativo ma utile)
# ------------------------------------------------------------
def extract_js_block(text: str) -> str:
    """
    Se il modello risponde con:
    ```javascript
    // codice
    ```
    estraggo solo il codice. Se non trova nulla, ritorno il testo originale ripulito.
    """
    if text is None:
        return ""
    pattern = r"```(?:javascript|js)?\s*(.*?)```"
    m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    if not m:
        return text.strip()
    return m.group(1).strip()

# ------------------------------------------------------------
# compute_metrics (ESATTAMENTE come mi hai dato)
# ------------------------------------------------------------
def compute_metrics(preds, refs, lang="javascript"):
    cb = []
    bleu = []
    meteor = []
    rouge1 = []
    rouge2 = []
    rougeL = []

    smoothie = SmoothingFunction().method1
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    for i in range(len(preds)):
        p = preds[i]
        r = refs[i]

        # tokenizzazione semplice a whitespace (per BLEU/METEOR)
        p_tok = p.split()
        r_tok = r.split()

        # ----- CodeBLEU per esempio -----
        res = calc_codebleu([r], [p], lang)
        cb.append(res["codebleu"])

        # ----- BLEU -----
        bleu_i = sentence_bleu(
            [r_tok],          # lista di riferimenti tokenizzati
            p_tok,            # ipotesi tokenizzata
            smoothing_function=smoothie,
        )
        bleu.append(bleu_i)

        # ----- METEOR -----
        meteor_i = meteor_score([r_tok], p_tok)
        meteor.append(meteor_i)

        # ----- ROUGE (lavora su stringhe intere) -----
        r_scores = rouge.score(r, p)
        rouge1.append(r_scores["rouge1"].fmeasure)
        rouge2.append(r_scores["rouge2"].fmeasure)
        rougeL.append(r_scores["rougeL"].fmeasure)

    # ----- CodeBERTScore F1 (lista) -----
    _, _, F1, _ = codebert_score(preds, refs, lang=lang)
    f1 = [float(x) for x in F1]

    return cb, f1, bleu, meteor, rouge1, rouge2, rougeL

# ------------------------------------------------------------
# SALVATAGGIO JSONL + RIEPILOGO
# ------------------------------------------------------------

def append_results_to_jsonl(new_rows, columns, path=OUT_JSONL):
    with open(path, "a", encoding="utf-8") as f:
        for row in new_rows:
            rec = dict(zip(columns, row))
            json.dump(rec, f, ensure_ascii=False)
            f.write("\n")

def print_partial_summary(results, columns):
    if not results:
        print("\n[Nessun risultato ancora]\n")
        return
    df = pd.DataFrame(results, columns=columns)
    summary = df.groupby(["model", "variant", "quant"])[
        ["codebleu", "codebert_f1"]
    ].mean()
    print("\n=== Partial summary ===")
    print(summary)
    print("=======================\n")

# ------------------------------------------------------------
# LM STUDIO: CLI (lms) PER CARICARE / SCARICARE MODELLI
# ------------------------------------------------------------
def lms_unload_all():
    """
    Esegue: lms unload --all
    """
    #print("[lms] unload --all")
    try:
        subprocess.run(
            ["lms", "unload", "--all"],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",      # <--- aggiunto
            errors="ignore",       # <--- per sicurezza
        )
    except FileNotFoundError:
        print("ERRORE: comando 'lms' non trovato. Aggiungi LM Studio CLI al PATH.")

def lms_load(model_key: str, identifier: str, gpu: str | None = None,
             context_length: int | None = None, ttl: int | None = None):
    cmd = ["lms", "load", model_key]
    if gpu is not None:
        cmd += ["--gpu", str(gpu)]
    if context_length is not None:
        cmd += ["--context-length", str(context_length)]
    if ttl is not None:
        cmd += ["--ttl", str(ttl)]

    #print(f"[lms] {' '.join(cmd)}")
    try:
        t0 = time.time()
        proc = subprocess.run(
            cmd,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",      # <--- aggiunto
            errors="ignore",       # <--- per sicurezza
        )
        dt = time.time() - t0
        #print(f"[lms] load completato in {dt:.1f}s")
        #print(proc.stdout)
    except FileNotFoundError:
        print("ERRORE: comando 'lms' non trovato. Aggiungi LM Studio CLI al PATH.")
        raise
    except subprocess.CalledProcessError as e:
        print("ERRORE nel caricamento modello LM Studio:")
        print(e.stdout)
        raise

# ------------------------------------------------------------
# LM STUDIO: chiamata REST /api/v0/chat/completions
# ------------------------------------------------------------
def lmstudio_chat_once(identifier: str, prompt: str,
                       max_new_tokens: int = 512,
                       temperature: float = 0.0,
                       top_p: float = 1.0):
    """
    Una chiamata a:
      POST /api/v0/chat/completions
    Ritorna (testo_generato, quant_label) dove quant_label viene da model_info.quant,
    se presente (es. "Q4_K_M", "Q8_0", ecc.).
    """
    url = LMSTUDIO_BASE_URL.rstrip("/") + "/api/v0/chat/completions"
    payload = {
        "model": identifier,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_new_tokens,
        "stream": False,
        "top_p": top_p,
    }
    r = requests.post(url, json=payload, timeout=600)
    r.raise_for_status()
    data = r.json()

    # testo generato
    text = data["choices"][0]["message"]["content"].strip()

    # quantizzazione (se disponibile)
    quant = None
    model_info = data.get("model_info", {})
    if isinstance(model_info, dict):
        quant = model_info.get("quant") or model_info.get("quantization")

    return text, quant

# ------------------------------------------------------------
# CONFIG MODELLI (CICLO A LA "for key in MODEL_REGISTRY")
# ------------------------------------------------------------
@dataclass
class LMStudioModelConfig:
    name: str
    model_key: str     # quello della lista "Select a model to load"
    identifier: str    # alias usato nelle API REST
    quant_hint: str
    gpu: str | None = None
    context_length: int | None = None
    ttl: int | None = None

MODEL_REGISTRY: list[LMStudioModelConfig] = [
    # -------------------------
    # Mamba-Codestral-7B (gabriellarson)
    # -------------------------
    LMStudioModelConfig(
        name="mamba_codestral_7b_q4_0",
        model_key="gabriellarson/Mamba-Codestral-7B-v0.1-GGUF/Mamba-Codestral-7B-v0.1-Q4_0.gguf",
        identifier="mamba-codestral-7b-v0.1-q4_0",
        quant_hint="Q4_0",
        gpu="max",
        context_length=4096,
    ),
    # -------------------------
    # CodeLlama-7B-Instruct (TheBloke)
    # -------------------------
    LMStudioModelConfig(
        name="codellama_7b_instruct_q4_k_m",
        model_key="TheBloke/CodeLlama-7B-Instruct-GGUF/codellama-7b-instruct.Q4_K_M.gguf",
        identifier="codellama-7b-instruct-q4_k_m",
        quant_hint="Q4_K_M",
        gpu="max",
        context_length=4096,
    ),
    # -------------------------
    # Qwen2.5-Coder-7B-Instruct (Qwen)
    # -------------------------
    LMStudioModelConfig(
        name="qwen2_5_coder_7b_q4_k_m",
        model_key="Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/qwen2.5-coder-7b-instruct-q4_k_m.gguf",
        identifier="qwen2.5-coder-7b-q4_k_m",
        quant_hint="Q4_K_M",
        gpu="max",
        context_length=4096,
    ),
    # -------------------------
    # DeepSeek-Coder-6.7B-Instruct (TheBloke)
    # -------------------------
    LMStudioModelConfig(
        name="deepseek_coder_6_7b_q4_k_m",
        model_key="TheBloke/deepseek-coder-6.7B-instruct-GGUF/deepseek-coder-6.7b-instruct.Q4_K_M.gguf",
        identifier="deepseek-coder-6.7b-q4_k_m",
        quant_hint="Q4_K_M",
        gpu="max",
        context_length=4096,
    ),
]

# MODEL_REGISTRY: list[LMStudioModelConfig] = [
#     # -------------------------
#     # tap_ft_gemma_merge - 8.5B (Gemma)
#     # -------------------------
#     LMStudioModelConfig(
#         name="tap_ft_gemma_merge_8_5b_q4_k_m",
#         model_key="LDES777/tap_ft_gemma_merge-Q4_K_M-GGUF/tap_ft_gemma_merge-q4_k_m-00001-of-00001.gguf",
#         identifier="tap-ft-gemma-merge-8.5b-q4_k_m",
#         quant_hint="Q4_K_M",
#         gpu="max",
#         context_length=4096,
#     ),
# 
#     # -------------------------
#     # tap_fc_deepseek_merge - 6.7B (Llama)
#     # -------------------------
#     LMStudioModelConfig(
#         name="tap_fc_deepseek_merge_6_7b_q4_k_m",
#         model_key="LDES777/tap_fc_deepseek_merge-Q4_K_M-GGUF/tap_fc_deepseek_merge-q4_k_m-00001-of-00002.gguf",
#         identifier="tap-fc-deepseek-merge-6.7b-q4_k_m",
#         quant_hint="Q4_K_M",
#         gpu="max",
#         context_length=4096,
#     ),
# 
#     # -------------------------
#     # tap_ft_codellama_merge - 7B (Llama)
#     # -------------------------
#     LMStudioModelConfig(
#         name="tap_ft_codellama_merge_7b_q4_k_m",
#         model_key="LDES777/tap_ft_codellama_merge-Q4_K_M-GGUF/tap_ft_codellama_merge-q4_k_m-00001-of-00002.gguf",
#         identifier="tap-ft-codellama-merge-7b-q4_k_m",
#         quant_hint="Q4_K_M",
#         gpu="max",
#         context_length=4096,
#     ),
# 
#     # -------------------------
#     # tap_ft_qwen_merge - 7.6B (Qwen2)
#     # -------------------------
#     LMStudioModelConfig(
#         name="tap_ft_qwen_merge_7_6b_q4_k_m",
#         model_key="LDES777/tap_ft_qwen_merge-Q4_K_M-GGUF/tap_ft_qwen_merge-q4_k_m-00001-of-00002.gguf",
#         identifier="tap-ft-qwen-merge-7.6b-q4_k_m",
#         quant_hint="Q4_K_M",
#         gpu="max",
#         context_length=4096,
#     ),
# 
#     # -------------------------
#     # tap_ft_codestral_merge - 7.3B (mamba2 / Codestral)
#     # -------------------------
#     LMStudioModelConfig(
#         name="tap_ft_codestral_merge_7_3b_q4_k_m",
#         model_key="LDES777/tap_ft_codestral_merge-Q4_K_M-GGUF/tap_ft_codestral_merge-q4_k_m-00001-of-00003.gguf",
#         identifier="tap-ft-codestral-merge-7.3b-q4_k_m",
#         quant_hint="Q4_K_M",
#         gpu="max",
#         context_length=4096,
#     ),
# ]

df = load_df_real()
prompts = df["instruct_ft"].tolist()
refs    = df["filter_code"].tolist()

columns = [
    "model", "variant", "quant", "row_id",
    "prompt", "reference", "generated",
    "codebleu", "codebert_f1", "bleu", "meteor",
    "rouge1", "rouge2", "rougeL"]

results = []
model_counter = 0

for cfg in MODEL_REGISTRY:
    model_counter += 1

    # 1) scarica tutto quello che c'è in memoria
    lms_unload_all()

    # 2) carica il modello con identifier fisso
    lms_load(
        model_key=cfg.model_key,
        identifier=cfg.identifier,
        gpu=cfg.gpu,
        context_length=cfg.context_length,
        ttl=cfg.ttl,
    )

    # 3) generazione su tutti i prompt
    preds_raw = []
    quant_seen = None

    for p in tqdm(prompts, desc=f"[{cfg.name}] generating"):
        out, quant = lmstudio_chat_once(
            identifier=cfg.identifier,
            prompt=p,
            max_new_tokens=512,
            temperature=0.0,
            top_p=1.0,
        )
        if quant_seen is None and quant is not None:
            quant_seen = quant
        preds_raw.append(out)

    # Pulizia opzionale: estraggo solo il blocco JS
    preds = [extract_js_block(x) for x in preds_raw]

    # 4) metriche (tutti insieme, come batch unico)
    cb, f1, bleu, meteor, r1, r2, rL = compute_metrics(preds, refs, lang=LANG)

    quant_label = quant_seen or cfg.quant_hint or "unknown"
    variant = "base"

    # 5) salvataggio JSONL (prompt + ref + output + metriche)
    new_rows = []
    for i, (prompt, ref, gen, c, f, b, m, rr1, rr2, rrL) in enumerate(
        zip(prompts, refs, preds, cb, f1, bleu, meteor, r1, r2, rL)
    ):
        new_rows.append([
            cfg.name,     # model
            variant,      # variant
            quant_label,  # quant (da REST o da hint)
            i,            # row_id
            prompt,
            ref,
            gen,
            c,
            f,
            b,
            m,
            rr1,
            rr2,
            rrL,
        ])

    append_results_to_jsonl(new_rows, columns)
    results.extend(new_rows)

    # 6) unload finale per lasciare la GPU pulita
    lms_unload_all()


Project root detected: C:\Users\ldesa\PycharmProjects\TAP basic
Loaded df_real shape=(351, 29)


[mamba_codestral_7b_q4_0] generating:   0%|          | 0/351 [00:00<?, ?it/s]

[codellama_7b_instruct_q4_k_m] generating:   0%|          | 0/351 [00:00<?, ?it/s]

[qwen2_5_coder_7b_q4_k_m] generating:   0%|          | 0/351 [00:00<?, ?it/s]

[deepseek_coder_6_7b_q4_k_m] generating:   0%|          | 0/351 [00:00<?, ?it/s]